# Event forecasting with MAPM

Event forecasting predicts what event comes next. MAPM records timestamped
non-fatal machine errors, so it supports a transparent first-order transition
baseline.

## Learning goals

- create next-event labels within each machine;
- fit a transition table on earlier events;
- compare the model with a global-majority baseline on later events.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "datasets").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import datasets
from sklearn.metrics import accuracy_score

In [ ]:
datasets.download("mapm")
errors = (
    datasets.load("mapm", table="errors")
    .sort("machineID", "datetime")
    .with_columns(
        pl.col("errorID").shift(-1).over("machineID").alias("next_error"),
        pl.int_range(pl.len()).over("machineID").alias("event_order"),
        pl.len().over("machineID").alias("event_count"),
    )
    .drop_nulls("next_error")
)
errors.select("machineID", "datetime", "errorID", "next_error").head()

In [ ]:
train = errors.filter(pl.col("event_order") < pl.col("event_count") * 0.8)
test = errors.filter(pl.col("event_order") >= pl.col("event_count") * 0.8)

transition = (
    train.group_by("errorID", "next_error")
    .len()
    .sort(["errorID", "len"], descending=[False, True])
    .group_by("errorID", maintain_order=True)
    .first()
    .select("errorID", pl.col("next_error").alias("predicted_error"))
)
majority_error = train["next_error"].mode()[0]
scored = test.join(transition, on="errorID", how="left").with_columns(
    pl.col("predicted_error").fill_null(majority_error)
)
print(
    "Transition accuracy:",
    f"{accuracy_score(scored['next_error'].to_numpy(), scored['predicted_error'].to_numpy()):.3f}",
)
print(
    "Majority accuracy:",
    f"{accuracy_score(scored['next_error'].to_numpy(), np.repeat(majority_error, scored.height)):.3f}",
)

In [ ]:
transition.sort("errorID")

## Interpretation and limitations

The transition baseline ignores elapsed time, telemetry, maintenance, and
longer history. A useful sequence model should improve on both this baseline
and the global majority while preserving a chronological, machine-aware split.